# Phase 121 — CryptoBench: Does Topology Help Cryptic-Pocket Prediction?

Thread 1 proved persistent homology significantly improves localization
on the main COACH420/HOLO4K pipeline (H1 > H0, topology overall beats
no-topology). Never tested on CryptoBench (a structurally different,
harder problem -- cryptic pockets are apo-INVISIBLE by definition, so
static-structure topology might carry less signal, or might matter MORE
since it captures cavity/void geometry that residue identity alone
can't). Trains full (with topology) and no_topo (topology zeroed) in the
SAME run, seed=0, for a direct comparison.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q torch-geometric==2.6.1

## Cell 2 — Setup, load Phase 117's blob (identical to Phase118/119)

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              matthews_corrcoef, f1_score, accuracy_score)

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase117_cryptobench_graphs_aux.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"])
POS_WEIGHT = float(blob["pos_weight"]); POS_WEIGHT_AUX = float(blob["pos_weight_aux"])
splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | pos_weight {POS_WEIGHT:.2f} | "
      f"pos_weight_aux {POS_WEIGHT_AUX:.2f} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

## Cell 3 — Model (identical to Phase118/119)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus the localization head,
    plus (new) an optional auxiliary "any-pocket" classification head --
    structurally identical to the main head, predicting a broader/easier
    superset target (y_aux: cryptic OR noncryptic pocket) to test whether
    it helps the shared trunk learn better intermediate features."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True, use_aux_head=False):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head; s.use_aux_head=use_aux_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))
        if use_aux_head:
            s.aux_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        aux_logits = s.aux_head(h).squeeze(-1) if s.use_aux_head else None
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center, aux_logits

## Cell 4 — Training machinery (identical to Phase119, no aux head, no masking -- this is Phase119's exact baseline arm, reused as the "full" topology arm here)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c, _ = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head, use_aux_head=False).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 5 — Train "full" (with topology) first while data is pristine, then zero d.x[:, N_PRE:] for "no_topo", same seed=0, same run

In [ ]:
BEST_LAMBDA = 0.1
results = []
t0 = time.time()

m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=0)
v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
torch.save(m.state_dict(), WORK/"cache/cryptobench_full_seed0.pt")
results.append(dict(arm="full (with topology)", val_pr=val_pr, val_dcc=v.get("dcc", float("nan"))))
print(f"  full (with topology)   seed0: val PR {val_pr:.4f}  val DCC {v.get('dcc', float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]")

for d in dataset:
    d.x[:, N_PRE:] = 0.0   # zero the topology block -- same technique as Thread 1's arm-masking
m2, val_pr2 = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=0)
v2 = run_epoch(m2, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
torch.save(m2.state_dict(), WORK/"cache/cryptobench_notopo_seed0.pt")
results.append(dict(arm="no_topo (topology zeroed)", val_pr=val_pr2, val_dcc=v2.get("dcc", float("nan"))))
print(f"  no_topo (zeroed)       seed0: val PR {val_pr2:.4f}  val DCC {v2.get('dcc', float('nan')):.4f}  [{(time.time()-t0)/60:.1f}m]")

print(f"\n--- topology ablation (single seed=0, same run) ---")
full_pr = results[0]["val_pr"]
for r in results:
    print(f"  {r['arm']:<30} val PR {r['val_pr']:.4f}  ({r['val_pr']-full_pr:+.4f} vs full)  val DCC {r['val_dcc']:.4f}")
json.dump(results, open(WORK/"phase121_topo_ablation_results.json", "w"), indent=2)
print("\nsaved -> phase121_topo_ablation_results.json")

---

## Reading this

- **full clearly beats no_topo** (gap outside typical single-seed noise,
  ~>0.01-0.015 val PR) -- topology's benefit generalizes to cryptic
  pockets too, a strong finding worth an 8-seed confirmation and
  reporting in any writeup.
- **Close/tied** -- topology's contribution is specific to the main
  pipeline's task/data scale, doesn't clearly transfer to this harder,
  smaller-data problem.
- **no_topo beats full** -- would be a genuinely surprising reversal,
  worth real scrutiny (e.g. is topology just adding noise on 700 chains
  when 5 dims might already be plenty of capacity for the trunk).